In Databricks, **`%fs ls`** is a file system command line shortcut used to **list the contents of a directory** inside your linked cloud storage.

It is the Databricks notebook equivalent of running `ls` in a Linux terminal or `dir` in Windows Command Prompt.

---

### 🔍 Breaking Down the Syntax

* **`%fs`**: This is a Databricks **magic command**. It tells the notebook cell: *"Hey, don't execute this as Python or SQL. Execute this directly using the Databricks File System (DBFS) utility layer."*
* **`ls`**: Short for "list".

---

### 🛠️ Real-World Code Examples

#### **1. List the root DBFS directory**

If you want to see the default system folders inside your workspace:

```bash
%fs ls /

```

#### **2. Inspect your custom Azure Data Lake (ADLS Gen2) folder**

Since you are mapping your own external containers for the course, you can use it to verify if your Parquet or Delta files actually landed on Azure:

```bash
%fs ls abfss://lab-data@delta0lake0lab0storageac.dfs.core.windows.net/invoices/

```

---

### 📊 What does the output look like?

When you run `%fs ls`, Databricks returns a clean, scannable table with three structural columns:

| path | name | size |
| --- | --- | --- |
| `dbfs:/invoices/invoices_1_100.parquet` | `invoices_1_100.parquet` | `59214` *(in bytes)* |
| `dbfs:/invoices/_delta_log/` | `_delta_log/` | `0` *(directories show as 0)* |

---

### 💡 Programmatic Alternative (Python/PySpark)

If you ever need to capture the file list inside a Python loop or store it in a variable (which magic commands cannot do), you drop the `%fs ls` shortcut and use the native production-grade **`dbutils`** package instead:

```python
# This does the exact same thing under the hood as %fs ls
files = dbutils.fs.ls("dbfs:/invoices/")
display(files)

```

You will see Afaque use `%fs ls` constantly throughout the course videos just to quickly check if a transaction created a new `.json` log or a fresh `.parquet` file back in your storage container!

In [0]:
%fs ls /

In [0]:
%fs ls abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/

In [0]:
files = dbutils.fs.ls("/")
display(files)
# Sample Parquet files used in this lab are stored in this location.
files = dbutils.fs.ls("abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/")
display(files)

In [0]:
%sql
-- 1. registering an External Location and Catalog in Unity Catalog.
-- (Note: Use the exact name of the Storage Credential you created earlier)
CREATE EXTERNAL LOCATION IF NOT EXISTS delta_lake_lab_adlsg2_ext_location_delta URL
'abfss://dbr-managed-tables-container@delta0lake0lab0storageac.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `delta0lake0lab0storageac_adls_sa_el_azuremanagedidentity_1783847538585`);

-- 2. Create the catalog pointing to your custom external location
CREATE CATALOG IF NOT EXISTS delta_catalog
MANAGED LOCATION 'abfss://dbr-managed-tables-container@delta0lake0lab0storageac.dfs.core.windows.net/';

-- 3. Create your schema
CREATE SCHEMA IF NOT EXISTS delta_catalog.delta_db;

In [0]:
%sql
-- Below path is also registered as an External Location in Unity Catalog, thats why its accessible.
SELECT
  *
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`
LIMIT 5;

In [0]:
%sql
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices AS --(CTAS)
SELECT
  *
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`;

In [0]:
%sql
DESCRIBE EXTENDED delta_catalog.delta_db.invoices;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices;

In [0]:
%sql
-- ALTER TABLE delta_catalog.delta_db.invoices_101_200
-- RENAME TO delta_catalog.delta_db.invoices;

-- DROP TABLE delta_catalog.delta_db.invoices;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices
  SELECT
    *
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
%sql
SELECT * FROM delta_catalog.delta_db.invoices; 

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices;

In [0]:
%sql
SELECT
  MIN(customer_id) AS min_value,
  MAX(customer_id) AS max_value,
  COUNT(*) AS total_rows
FROM
  delta_catalog.delta_db.invoices

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices
WHERE
  customer_id = 1;

In [0]:
%sql
UPDATE
  delta_catalog.delta_db.invoices
SET
  quantity = 10
WHERE
  customer_id = 1;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices
WHERE
  customer_id = 99;

In [0]:
%sql
DELETE FROM
  delta_catalog.delta_db.invoices
WHERE
  customer_id = 99;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices
  SELECT
    *
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_201_99457.parquet`

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices;

In [0]:
%sql
SELECT
  count(*) as total_rows
FROM
  delta_catalog.delta_db.invoices;

In [0]:
# display(_sqldf)